# Evaluating Time Series Foundation Models for Tropical Solar Irradiance Forecasting

**Target:** PREE 2026 (IEEE Xplore) — Tokyo, Oct 3-6, 2026  
**Deadline:** May 15, 2026

This notebook runs the entire research pipeline step-by-step.

## Pipeline Overview

| Step | Script | What it does |
|------|--------|-------------|
| 1 | `data/download_nasa_power.py` | Download 6 years of hourly solar data for Laguna de Bay |
| 2 | `data/prepare_data.py` | Split into train/val/test, create forecast windows |
| 3 | `figures/eda.py` | Exploratory analysis and seasonal pattern figures |
| 4 | `eval/zero_shot.py` | Zero-shot evaluation of 6 foundation models |
| 5 | `eval/baselines.py` | Traditional baselines (XGBoost, LSTM, Persistence) |
| 6 | `finetune/chronos_ft.py` | Fine-tune Chronos Small + Base |
| 7 | `finetune/ttm_ft.py` | Fine-tune TTM-R2 |
| 8 | `eval/finetuned.py` | Fine-tuned Chronos vs zero-shot |
| 9 | `eval/all_finetuned.py` | Full comparison + CRPS + DM significance tests |
| 10 | `eval/ablation.py` | Ablation: training steps vs performance |
| 11 | `experiments/data_efficiency.py` | Training data size experiment |
| 12 | `experiments/sensitivity_analysis.py` | Hyperparameter sweep (27 configs) |
| 13 | `figures/generate.py` | Generate all paper figures |

## Models Evaluated

**Foundation Models (zero-shot + fine-tuned):**
- Chronos-2 (Amazon) — 120M params, encoder-only (2025)
- Chronos-T5-Small (Amazon) — ~60M params (fine-tuned)
- Chronos-T5-Base (Amazon) — ~200M params (fine-tuned)
- TimesFM 2.5 (Google) — 200M params, patched decoder (2026)
- Moirai 2.0 Small (Salesforce) — ~14M params, decoder-only (2025)
- TTM-R2 (IBM) — ~1M params, MLP-Mixer (fine-tuned) (2025)

**Traditional Baselines:**
- Persistence (naive: repeat last day)
- XGBoost with engineered features
- LSTM (2-layer, 64 hidden units)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
os.chdir(os.path.abspath(".."))
print(f"Working directory: {os.getcwd()}")

## Step 1 — Download NASA POWER Data

Downloads hourly solar irradiance and weather data for Laguna de Bay (14.3833°N, 121.2500°E), 2020–2025.  
**Output:** `data/raw/nasa_power_laguna_de_bay_2020_2025.csv` (~52,600 records)

Skip this cell if data is already downloaded.

In [ ]:
from pathlib import Path

data_file = Path("data/raw/nasa_power_laguna_de_bay_2020_2025.csv")
if data_file.exists():
    print(f"Data already exists: {data_file} ({data_file.stat().st_size / 1e6:.1f} MB)")
    print("Skipping download. Delete the file to re-download.")
else:
    %run ../src/data/download_nasa_power.py

## Step 2 — Prepare Data

Cleans data, splits by year, creates sliding-window forecast samples.

- **Train:** 2020–2023 (for fine-tuning)
- **Validation:** 2024
- **Test:** 2025
- **Context:** 168 hours (7 days) → **Predict:** 24h and 72h

**Output:** `data/processed/` — CSVs, `.npz` forecast windows, training series

In [ ]:
%run ../src/data/prepare_data.py

## Step 3 — Exploratory Data Analysis

Generates 5 figures showing seasonal patterns, wet vs dry contrast, cloud cover effects, and inter-annual variability.

**Output:** `results/figures/fig1_*.png` through `fig5_*.png`

In [ ]:
%run ../src/figures/eda.py

In [ ]:
# Preview EDA figures
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path("results/figures")
for fig in sorted(fig_dir.glob("fig[1-5]_*.png")):
    print(f"\n--- {fig.name} ---")
    display(Image(filename=str(fig), width=700))

## Step 4 — Zero-Shot Foundation Model Evaluation

Runs **6 foundation models** on the test set **without any fine-tuning**.

**Models:**
| Model | Source | Params | Architecture |
|-------|--------|--------|-------------|
| Chronos-2 | Amazon | 120M | Encoder-only |
| Chronos-T5-Small | Amazon | ~60M | T5 enc-dec |
| Chronos-T5-Base | Amazon | ~200M | T5 enc-dec |
| TimesFM 2.5 | Google | 200M | Patched decoder |
| Moirai 2.0 Small | Salesforce | ~14M | Decoder-only |
| TTM-R2 | IBM | ~1M | MLP-Mixer |

Plus **Persistence baseline** (naive repeat).

**Metrics:** MAE, RMSE, MASE, Skill Score — broken down by overall / dry / wet season

**Output:** `results/tables/zero_shot_results.csv`, prediction `.npy` files, logs in `results/logs/`

**Note:** Models are downloaded from HuggingFace on first run. If a model fails to load (missing dependency), it is skipped gracefully.

In [ ]:
%run ../src/eval/zero_shot.py

In [ ]:
# Review zero-shot results
import pandas as pd

zs = pd.read_csv("results/tables/zero_shot_results.csv")
print("=== Zero-Shot Results ===\n")
print(zs.to_markdown(index=False))

## Step 5 — Traditional Baselines

Trains and evaluates XGBoost (with engineered features) and LSTM on the same test windows.

**Models:**
- **XGBoost:** Direct multi-step forecasting. Features: last 72h raw values, daily stats (mean/std/max), daytime-only stats
- **LSTM:** 2-layer, 64 hidden units, 50 epochs, trained on normalized data

**Output:** `results/tables/baseline_results.csv`

In [ ]:
%run ../src/eval/baselines.py

In [ ]:
# Review baseline results
bl = pd.read_csv("results/tables/baseline_results.csv")
print("=== Baseline Results ===\n")
print(bl.to_markdown(index=False))

## Step 6 — Fine-tune Chronos on Tropical Solar Data

Fine-tunes both `chronos-t5-small` and `chronos-t5-base` on 4 years (2020–2023) of Laguna de Bay solar data.

- Training data is split into monthly chunks + full-year series
- 5000 training steps, learning rate 5e-5, cosine schedule
- Uses model's native context and prediction lengths

**Output:** `models/ft-chronos-t5-small/` and `models/ft-chronos-t5-base/`

**Note:** This step takes ~30–60 minutes depending on hardware.

In [ ]:
%run ../src/finetune/chronos_ft.py

## Step 7 — Fine-tune TTM-R2

Fine-tunes IBM's TTM-R2 (MLP-Mixer, ~1M params) using standard HuggingFace Trainer.

**Output:** `models/ft-ttm-r2/`

In [ ]:
%run ../src/finetune/ttm_ft.py

## Step 8 — Evaluate Fine-tuned Chronos vs Zero-shot

Compares fine-tuned Chronos Small and Base against their zero-shot versions. Computes improvement percentage per season.

**Output:** `results/tables/finetuned_comparison.csv`

In [ ]:
%run ../src/eval/finetuned.py

In [ ]:
# Review fine-tuned comparison
ft = pd.read_csv("results/tables/finetuned_comparison.csv")
print("=== Fine-tuned vs Zero-shot ===\n")
print(ft.to_markdown(index=False))

## Step 9 — Full Model Comparison + Statistical Tests

Comprehensive evaluation of all models (zero-shot, fine-tuned, baselines) with CRPS probabilistic metrics and Diebold-Mariano pairwise significance tests.

**Output:** `results/tables/all_models_comparison.csv`, `results/tables/dm_tests_*.csv`

In [ ]:
%run ../src/eval/all_finetuned.py

In [ ]:
# Review full comparison
allm = pd.read_csv("results/tables/all_models_comparison.csv")
print("=== All Models Comparison ===\n")
print(allm.to_markdown(index=False))

## Step 10 — Ablation Study: Training Steps vs Performance

Evaluates how performance changes with different numbers of fine-tuning steps.

**Output:** `results/tables/ablation_steps.csv`

In [ ]:
%run ../src/eval/ablation.py

In [ ]:
# Review ablation results
abl = pd.read_csv("results/tables/ablation_steps.csv")
print("=== Ablation Study ===\n")
print(abl.to_markdown(index=False))

## Step 11 — Data Efficiency Experiment

How many months of local data are needed before fine-tuning pays off?

Fine-tunes Chronos with: 3, 6, 12, 24, and 48 months of training data, then evaluates each.

**Output:** `results/tables/data_efficiency_results.csv`

**Note:** This is a long step — runs 5 fine-tuning jobs sequentially. Expect ~1-2 hours.

In [ ]:
%run ../src/experiments/data_efficiency.py

In [ ]:
# Review data efficiency results
de = pd.read_csv("results/tables/data_efficiency_results.csv")
print("=== Data Efficiency ===\n")
print(de.to_markdown(index=False))

## Step 12 — Sensitivity Analysis (Hyperparameter Sweep)

Sweeps fine-tuning hyperparameters to find the best config.

**Hyperparameter Grid (27 combinations):**

| Parameter | Values |
|-----------|--------|
| Learning rate | 1e-3, 1e-4, 1e-5 |
| Training steps | 500, 1000, 2000 |
| Context length | 72h (3 days), 168h (7 days), 336h (14 days) |

**Output:** `results/tables/sensitivity_results.csv`

**Note:** Computationally expensive (~27 fine-tuning runs). Expect several hours.

In [ ]:
%run ../src/experiments/sensitivity_analysis.py

In [ ]:
# Review sensitivity analysis — best configs
sa = pd.read_csv("results/tables/sensitivity_results.csv")
print("=== Sensitivity Analysis ===\n")

for horizon in ["24h", "72h"]:
    for season in ["all", "wet"]:
        subset = sa[(sa["horizon"] == horizon) & (sa["season"] == season)]
        if subset.empty:
            continue
        best = subset.loc[subset["MAE"].idxmin()]
        print(f"Best for {horizon} ({season}): lr={best['learning_rate']}, "
              f"steps={int(best['max_steps'])}, ctx={int(best['context_length'])} "
              f"→ MAE={best['MAE']:.2f}")

print("\n--- Full Results ---\n")
print(sa.to_markdown(index=False))

## Step 13 — Generate All Paper Figures

Creates publication-quality figures from all saved results.

**Output:** `results/figures/fig_*.png`

In [ ]:
%run ../src/figures/generate.py

In [ ]:
# Preview all paper figures
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path("results/figures")
for fig in sorted(fig_dir.glob("fig_*.png")):
    print(f"\n--- {fig.name} ---")
    display(Image(filename=str(fig), width=700))

## Summary — Combined Results

Load all result tables and display the full picture.

In [ ]:
import pandas as pd
import json
from pathlib import Path

table_dir = Path("results/tables")
log_dir = Path("results/logs")

# Combine all result CSVs
all_dfs = []
for csv in sorted(table_dir.glob("*.csv")):
    df = pd.read_csv(csv)
    df["source"] = csv.stem
    all_dfs.append(df)
    print(f"Loaded: {csv.name} ({len(df)} rows)")

if all_dfs:
    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv(table_dir / "all_results_combined.csv", index=False)
    
    # Save as JSON for further analysis
    combined.to_json(log_dir / "all_results_combined.json", orient="records", indent=2)
    
    print(f"\n=== Combined Results ({len(combined)} rows) ===\n")
    print(combined.to_markdown(index=False))
else:
    print("No result files found. Run the pipeline steps above first.")